In [19]:
import xarray as xr
from dask.distributed import Client
from dask_jobqueue import PBSCluster
import matplotlib.pyplot as plt
import numpy as np

In [3]:
cluster = PBSCluster(
    cores=1,
    memory='40GB',
    processes=1,
    queue='casper',
    local_directory='$TMPDIR',
    account='P93300313',
    walltime='4:00:00'
)
cluster.scale(jobs=10)
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.183:45861,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


# data i

In [13]:
path = '/glade/work/acruz/Caribbean_Heat_data/ERA5/'
hi_ds = xr.open_dataset(path+'hourly_HI.nc')
hi_ds

<xarray.Dataset> Size: 30GB
Dimensions:    (time: 755304, latitude: 82, longitude: 121)
Coordinates:
  * time       (time) datetime64[ns] 6MB 1940-01-01 ... 2026-02-28T23:00:00
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0
Data variables:
    HI_hourly  (time, latitude, longitude) float32 30GB ...

In [14]:
rhanom_ds = xr.open_zarr(path+'ERA5_rh_anom')
rhanom_ds = rhanom_ds['__xarray_dataarray_variable__']
rhanom_ds.rename('RH')

<xarray.DataArray 'RH' (time: 755304, latitude: 82, longitude: 121)> Size: 30GB
dask.array<open_dataset-__xarray_dataarray_variable__, shape=(755304, 82, 121), dtype=float32, chunksize=(34332, 82, 121), chunktype=numpy.ndarray>
Coordinates:
  * time       (time) datetime64[ns] 6MB 1940-01-01 ... 2026-02-28T23:00:00
    month      (time) int64 6MB dask.array<chunksize=(34332,), meta=np.ndarray>
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0

In [15]:
tanom_ds = xr.open_zarr(path+'ERA5_t_anom')
tanom_ds

<xarray.Dataset> Size: 30GB
Dimensions:    (time: 755304, latitude: 82, longitude: 121)
Coordinates:
  * time       (time) datetime64[ns] 6MB 1940-01-01 ... 2026-02-28T23:00:00
    month      (time) int64 6MB dask.array<chunksize=(34332,), meta=np.ndarray>
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0
Data variables:
    VAR_2T     (time, latitude, longitude) float32 30GB dask.array<chunksize=(34332, 82, 121), meta=np.ndarray>

# get max hi time stamps

In [16]:
# chunk for dask
hi_ds = hi_ds.chunk(time=17166)
tanom_ds = tanom_ds['VAR_2T'].chunk(time=17166)
rhanom_ds = rhanom_ds.chunk(time=17166)
tanom_ds

<xarray.DataArray 'VAR_2T' (time: 755304, latitude: 82, longitude: 121)> Size: 30GB
dask.array<rechunk-merge, shape=(755304, 82, 121), dtype=float32, chunksize=(17166, 82, 121), chunktype=numpy.ndarray>
Coordinates:
  * time       (time) datetime64[ns] 6MB 1940-01-01 ... 2026-02-28T23:00:00
    month      (time) int64 6MB dask.array<chunksize=(17166,), meta=np.ndarray>
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0

In [17]:
# make hour dim lazily
hi_ds = hi_ds.coarsen(time=24, boundary='trim').construct(time=('day', 'hour'))
tanom_ds = tanom_ds.coarsen(time=24, boundary='trim').construct(time=('day', 'hour'))
rhanom_ds = rhanom_ds.coarsen(time=24, boundary='trim').construct(time=('day', 'hour'))
tanom_ds

<xarray.DataArray 'VAR_2T' (day: 31471, hour: 24, latitude: 82, longitude: 121)> Size: 30GB
dask.array<reshape, shape=(31471, 24, 82, 121), dtype=float32, chunksize=(715, 24, 82, 121), chunktype=numpy.ndarray>
Coordinates:
    month      (day, hour) int64 6MB dask.array<chunksize=(715, 24), meta=np.ndarray>
    time       (day, hour) datetime64[ns] 6MB 1940-01-01 ... 2026-02-28T23:00:00
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0
Dimensions without coordinates: day, hour

In [18]:
hidmax_idx = hi_ds['HI_hourly'].argmax(dim='hour')
hidmax_idx

<xarray.DataArray 'HI_hourly' (day: 31471, latitude: 82, longitude: 121)> Size: 2GB
dask.array<nanarg_agg-aggregate, shape=(31471, 82, 121), dtype=int64, chunksize=(715, 82, 121), chunktype=numpy.ndarray>
Coordinates:
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0
Dimensions without coordinates: day

In [20]:
def sel_at_idx(ds, idx_ds):
    # add hour dimension to indexing array
    idx_ds = np.expand_dims(idx_ds, axis=-1)
    # select along hour axis and remove hour axis
    return np.take_along_axis(ds, idx_ds, axis=-1).squeeze(axis=-1)

In [27]:
tanom_peakHI = xr.apply_ufunc(sel_at_idx, tanom_ds, hidmax_idx,
                              input_core_dims=[['hour'], []],
                              output_core_dims=[[]],
                              dask='parallelized',
                              output_dtypes=[tanom_ds.dtype]
                             )
tanom_peakHI = tanom_peakHI.rename('T_anom_HIdmax')
tanom_peakHI

<xarray.DataArray 'T_anom_HIdmax' (day: 31471, latitude: 82, longitude: 121)> Size: 1GB
dask.array<transpose, shape=(31471, 82, 121), dtype=float32, chunksize=(715, 82, 121), chunktype=numpy.ndarray>
Coordinates:
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0
Dimensions without coordinates: day

In [28]:
rhanom_peakHI = xr.apply_ufunc(sel_at_idx, rhanom_ds, hidmax_idx,
                               input_core_dims=[['hour'], []],
                               output_core_dims=[[]],
                               dask='parallelized',
                               output_dtypes=[rhanom_ds.dtype]
                              )
rhanom_peakHI = rhanom_peakHI.rename('RH_anom_HIdmax')
rhanom_peakHI

<xarray.DataArray 'RH_anom_HIdmax' (day: 31471, latitude: 82, longitude: 121)> Size: 1GB
dask.array<transpose, shape=(31471, 82, 121), dtype=float32, chunksize=(715, 82, 121), chunktype=numpy.ndarray>
Coordinates:
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0
Dimensions without coordinates: day

In [29]:
tanom_peakHI.to_netcdf(path+'T_anom_during_peak_HI.nc')

In [30]:
rhanom_peakHI.to_netcdf(path+'RH_anom_during_peak_HI.nc')

In [11]:
client.shutdown()